# AutiLens — Phase 3: LoRA fine-tuning on A100

Phase 1 (frozen backbone + cached features) runs locally in minutes and does **not**
belong here. This notebook is only for LoRA, where the backbone weights change and
features can no longer be cached.

**Before running:** upload the packed cache to your **own private Google Drive**.
Build it locally with:
```bash
python -m scripts.pack_for_colab --out data/packed          # ~430 MB
python -m scripts.pack_for_colab --validate --out data/packed   # GATE — must PASS
```

> **Privacy.** This is video of real children sourced from YouTube/Facebook.
> Private Drive only — never GitHub, never a public dataset host (guide §14).

**Methodology carries over unchanged:** nested CV, thresholds/calibrators fitted on
the inner split and frozen before the outer fold is scored, and the 92-clip test set
is never touched here — selection is on outer OOF only.


> **Format note.** The cache ships as **WEBP lossless** (~1.0 GB), not JPEG.
> JPEG q95 was 0.32 GB and looked clean at 42.5 dB PSNR, but it shifted model
> probabilities by up to 0.218 and flipped a thresholded decision on **4 of 40
> clips** — enough to confound the small accuracy delta this phase measures.
> WEBP lossless decodes bit-exact (0 flips), so the extra 0.7 GB buys a result
> you can actually trust.


## 1. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Clone the repo (code only — no data in git)


In [ ]:
import os
REPO = 'https://github.com/YOUR_USERNAME/AutiLens.git'   # <-- set this
if not os.path.exists('AutiLens'):
    !git clone $REPO AutiLens
%cd AutiLens
!pip -q install librosa opencv-python-headless pyyaml tqdm scikit-learn


## 3. Mount Drive and unpack

Expects `windows_webp.tar` and `mel_f16.npz` in the Drive folder below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PACK = '/content/drive/MyDrive/autilens_packed'   # <-- set this
!mkdir -p data/processed/windows_raw
!tar -xf $PACK/windows_webp.tar -C data/processed/windows_raw
!cp $PACK/mel_f16.npz data/processed/
!du -sh data/processed/windows_raw


In [ ]:
# mel_f16.npz is one archive; the pipeline wants per-clip .npy, and the
# AUDIO FEATURE cache (ResNet18 over each mel) is what cv_lora actually reads.
!python -m scripts.pack_for_colab --unpack-mel --out $PACK
!python -m src.preprocessing.extract_features --only audio


## 4. Re-run the packed-cache gate on this machine

The gate was run locally, but decode paths differ between machines. If any
thresholded decision flips here, stop and use the raw cache.


In [ ]:
# The gate ran locally, but decode paths differ between machines.
!python -m scripts.pack_for_colab --validate --out $PACK --format webp --n 40 \
    || echo 'GATE FAILED - stop and use the raw cache'


## 5. Confirm the LoRA adapter is wired correctly

torchvision's Swin3D reads `qkv.weight` directly and passes it to a functional, so an
adapter overriding `forward()` silently does nothing. The audit below is what catches
that: base weights frozen, only A/B trainable.


In [ ]:
!python -m src.cv_lora --audit-only --stages last --rank 8
!python -m src.cv_lora --audit-only --stages all  --rank 8


## 6. Train

Start with variant **A** (`--stages last`, ~74K trainable). Expand only if it helps —
335 training clips is very little for a 28M-param backbone.

| variant | stages | rank | trainable | ~cost on A100 |
|---|---|---|---|---|
| A | last | 8 | 74 K | ~10 min |
| B | all | 8 | 212 K | ~20 min |
| C | all | 16 | 424 K | ~25 min |


In [ ]:
# Variant A. No --eval-test: selection is on outer OOF only.
# ~75s per fold-epoch on MPS; an A100 should be several times faster.
!python -m src.cv_lora --target families --stages last --rank 8 \
    --epochs 12 --batch-size 8 --tag lora_A


In [ ]:
# Variants B and C, only if A shows promise.
# !python -m src.cv_lora --target families --stages all --rank 8  --epochs 12 --batch-size 8 --tag lora_B
# !python -m src.cv_lora --target families --stages all --rank 16 --epochs 12 --batch-size 8 --tag lora_C


## 7. Compare — on OOF, never on test


In [ ]:
!python -m src.evaluation.summary


## 8. Copy results back to Drive

Checkpoints and reports only; no source video leaves this session.


In [ ]:
!mkdir -p $PACK/results
!cp -v models/lora_*.pt models/lora_*_oof.npz reports/lora_*_cv.json $PACK/results/ 2>/dev/null
!ls -la $PACK/results


## 9. Watch for overfitting

If inner-val macro-F1 runs far ahead of outer OOF, the adapter is memorising 335
clips — drop the rank or the epoch count. A LoRA variant that fails to beat the
frozen model is a valid, reportable result: Phase 4 gates on measurement, so nothing
weak ships.
